# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [3]:
# TODO: Create a .env file with the following variables
# OPENAI_API_KEY="YOUR_KEY"
# CHROMA_OPENAI_API_KEY="YOUR_KEY"
# TAVILY_API_KEY="YOUR_KEY"

In [ ]:
# Load environment variables
load_dotenv(override=True)
openai_key = os.getenv('OPENAI_API_KEY') #not using this key for now
tavily_key = os.getenv('TAVILY_API_KEY')

print(openai_key)  # This will print your OpenAI API key
print(tavily_key)  # This will print your Tavily API key

voc-128284763016886546721826946fbfde31796.64664478
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


### VectorDB Instance

### Collection

In [38]:
from openai import OpenAI
class VocareumEmbeddingFunction:
    def __init__(self, api_key):
        self.client = OpenAI(
            api_key=api_key,
            base_url='https://openai.vocareum.com/v1'
        )

    def __call__(self, input):
        if isinstance(input, str):
            input = [input]

        response = self.client.embeddings.create(
            model='text-embedding-3-small',
            input=input
        )
        return [item.embedding for item in response.data]

    def name(self):
        return "openai_embedding"
    
chroma_client = chromadb.PersistentClient(path="chromadb")
embedding_fn = VocareumEmbeddingFunction(openai_key)
chroma_client.delete_collection("udaplay")

collection = chroma_client.create_collection(
    name="udaplay",
    embedding_function=embedding_fn
)

### Add documents

In [39]:
# Make sure you have a directory "project/starter/games"
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # You can change what text you want to index
    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']} - {game['Publisher']}"

    # Use file name (like 001) as ID
    doc_id = os.path.splitext(file_name)[0]

    collection.add(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )
print(f"✓ Loaded {collection.count()} games from vector database")
ids = collection.get()['ids']
print(ids)

✓ Loaded 15 games from vector database
['001', '002', '003', '004', '005', '006', '007', '008', '009', '010', '011', '012', '013', '014', '015']
